# Lab 4.2: Context A/B/C Scorecard Evaluator

**Day 1 - Topic 4**

Goal: evaluate three context configurations against the same coding task using mandatory correctness, instruction adherence, and safety gates before comparing token and turn efficiency.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/context-claude-companion/blob/main/day1/lab4_2_context_ab_evaluator/start/lab4_2_context_ab_evaluator.ipynb)

In [ ]:
# No extra packages needed: this lab uses the Python standard library only.
# If running locally with the course venv already activated, you can skip this cell.

In [ ]:
def check(label: str, ok: bool) -> None:
    """Print PASS when a result is correct, TODO while a step is unfinished."""
    print(("PASS  " if ok else "TODO  ") + label)

## Section 1: Three labeled variants

Three synthetic benchmark records for the same coding task: a bloated and conflicting baseline (A), an over-trimmed "token miser" (B), and an engineered scoped context (C). The numbers are labeled synthetic. Replace them with your own `/context` and `/cost` readings when you run a real comparison.

In [ ]:
SYNTHETIC_BENCHMARK_VARIANTS = [
    {
        "id": "Variant A (Bloated & Conflicting)",
        "evidence_type": "synthetic",
        "commit": "a91f02b",
        "tests_passed": 4,
        "tests_total": 5,
        "invariants": {
            "uses_httpx_async_client": False,
            "preserves_timeout_config": True,
            "structured_audit_log": True,
        },
        "safety_gate_passed": True,
        "context_checkpoint_tokens": 24_800,
        "turns": 6,
        "session_cost_usd": None,  # Recorded as N/A when account mode does not expose dollar billing
    },
    {
        "id": "Variant B (Over-Trimmed 'Token Miser')",
        "evidence_type": "synthetic",
        "commit": "a91f02b",
        "tests_passed": 3,
        "tests_total": 5,
        "invariants": {
            "uses_httpx_async_client": True,
            "preserves_timeout_config": False,
            "structured_audit_log": False,
        },
        "safety_gate_passed": False,  # Omitted tenant-isolation check
        "context_checkpoint_tokens": 6_200,
        "turns": 2,
        "session_cost_usd": None,
    },
    {
        "id": "Variant C (Engineered Scoped Context)",
        "evidence_type": "synthetic",
        "commit": "a91f02b",
        "tests_passed": 5,
        "tests_total": 5,
        "invariants": {
            "uses_httpx_async_client": True,
            "preserves_timeout_config": True,
            "structured_audit_log": True,
        },
        "safety_gate_passed": True,
        "context_checkpoint_tokens": 11_400,
        "turns": 3,
        "session_cost_usd": None,
    },
]

for v in SYNTHETIC_BENCHMARK_VARIANTS:
    print(f"{v['id']:<42} tests {v['tests_passed']}/{v['tests_total']}  tokens {v['context_checkpoint_tokens']:>6,d}  turns {v['turns']}")

## Section 2: Apply the decision rule

A variant is eligible only when it passes the acceptance tests, follows every project invariant, and passes the safety gate. Token and turn counts are compared only after that.

In [ ]:
def evaluate_variant(v: dict) -> dict:
    """Apply the strict quality-first decision rule to a context evaluation record."""
    # TODO: tests_ok is True when v["tests_passed"] == v["tests_total"]
    # TODO: adherence_ok is True when every value in v["invariants"] is True  (hint: all(...))
    # TODO: safety_ok is bool(v["safety_gate_passed"])
    tests_ok = adherence_ok = safety_ok = False  # placeholders until you implement the three gates
    all_primary_gates_pass = tests_ok and adherence_ok and safety_ok

    if not tests_ok:
        reason = "REJECTED (failed acceptance tests)"
    elif not safety_ok:
        reason = "REJECTED (failed mandatory safety/tenant boundary gate)"
    elif not adherence_ok:
        reason = "REJECTED (failed project instruction adherence)"
    else:
        reason = "ELIGIBLE (passes all primary gates; compare efficiency)"

    cost_display = f"${v['session_cost_usd']:.2f}" if v["session_cost_usd"] is not None else "N/A"
    adherence_ratio = f"{sum(1 for ok in v['invariants'].values() if ok)}/{len(v['invariants'])}"

    return {
        "id": v["id"],
        "evidence": v["evidence_type"],
        "tests": f"{v['tests_passed']}/{v['tests_total']}",
        "adherence": adherence_ratio,
        "safety": "PASS" if safety_ok else "FAIL",
        "context_tokens": v["context_checkpoint_tokens"],
        "turns": v["turns"],
        "cost": cost_display,
        "eligible": all_primary_gates_pass,
        "verdict": reason,
    }

## Section 3: Select the winner

Among the eligible variants, pick the one with the smallest context footprint.

In [ ]:
def select_winner(evaluated: list[dict]) -> dict | None:
    """Among variants that pass every primary gate, pick the one with the fewest context tokens."""
    # TODO: Keep only the rows where row["eligible"] is True
    # TODO: Return the eligible row with the smallest context_tokens  (hint: min(..., key=lambda r: r["context_tokens"]))
    # TODO: Return None when no variant is eligible
    return None

## Section 4: Run the scorecard

Evaluate all three variants and print the scorecard.

In [ ]:
print("=== Controlled Context Evaluation Scorecard (Evidence Type: synthetic) ===")
print(
    f"{'Variant':<40} | {'Tests':<5} | {'Rules':<5} | {'Safety':<6} | "
    f"{'/context':>9} | {'Turns':>5} | {'/cost':>5} | {'Verdict'}"
)
print("-" * 122)

evaluated = [evaluate_variant(v) for v in SYNTHETIC_BENCHMARK_VARIANTS]
for row in evaluated:
    print(
        f"{row['id']:<40} | {row['tests']:<5} | {row['adherence']:<5} | {row['safety']:<6} | "
        f"{row['context_tokens']:>9,d} | {row['turns']:>5d} | {row['cost']:>5} | {row['verdict']}"
    )

winner = select_winner(evaluated)
print(f"\nSelected Configuration: {winner['id'] if winner else 'None'}")
print(
    "Notice: Variant B used the fewest tokens (6,200) and turns (2) but was automatically rejected "
    "because it failed acceptance tests and safety gates."
)

# Expected: A and B are REJECTED, C is ELIGIBLE and selected.
print()
check("Variants A and B are rejected, C is eligible", [r["eligible"] for r in evaluated] == [False, False, True])
check("Variant B has the lowest token count of all three", min(evaluated, key=lambda r: r["context_tokens"])["id"].startswith("Variant B"))
check("Variant C is selected even though B used fewer tokens", winner is not None and winner["id"].startswith("Variant C"))

In [ ]:
print("Lab complete.")
print("Takeaway: Always gate context changes on task correctness, instruction adherence, and safety first; compare /context, /cost, and turn counts only among passing configurations.")